# 3장 1강: CNN의 구조와 원리 — 실습문제

## 실습 목표

- 작은 필터를 이동하며 곱하고 더하는 합성곱 계산을 이해합니다.
- 같은 입력에서도 필터에 따라 다른 특징 맵이 만들어지는 이유를 설명합니다.
- 풀링으로 특징 맵을 요약하고, 남는 정보와 사라지는 정보를 구분합니다.
- CNN 기반 이미지 분류의 처리 순서와 각 층의 역할을 설명합니다.

## 실습 환경 / 데이터

- Python, pandas, NumPy를 사용합니다. 별도의 딥러닝 프레임워크는 필요하지 않습니다.
- **`jena_climate_2009_2016.csv`를 노트북과 같은 폴더에 두고**, 해당 폴더를 VS Code에서 열어 실행하세요. pandas와 NumPy가 설치된 Python 환경을 커널로 선택합니다.
- 제공된 파일은 **420,551행, 15열**입니다. 이번에는 **첫 16행**을 읽고, 그중 `Date Time`과 기온 열 `T (degC)`를 사용합니다. 기온 단위는 섭씨(℃)입니다.
- 실습은 첫 8개 관측값, 과제는 다음 8개 관측값을 사용합니다. 선택한 구간의 관측 간격은 10분입니다.
- 코드 셀은 준비 셀부터 위에서 아래로 실행하세요. `np.round(..., 4)`는 출력만 소수 넷째 자리까지 표시하며, 계산은 원래 값으로 합니다.

## 이번 강의의 범위

제공된 **3장 1강 목차: Convolution·Filter·Feature Map, Pooling과 특징 추출, CNN 기반 이미지 분류 구조**를 기준으로 구성했습니다. 강의 본문은 별도로 제공되지 않았으므로 필터 값과 크기 예시는 개념 확인을 위해 정한 값입니다.

Jena Climate는 기상 시계열 데이터입니다. **시간 방향의 1차원 합성곱**으로 원리를 계산하고, 이미지의 **가로·세로 방향 2차원 합성곱**과 연결합니다. 연속 기온값을 임의로 이미지처럼 바꾸지는 않습니다. 이미지 분류는 구조와 역할을 묻는 개념 문제이며, 실제 이미지 모델을 학습하거나 분류 정확도를 측정하는 실습은 아닙니다.

필터는 계산을 확인하기 위해 고정해서 제공합니다. 실제 CNN에서는 일반적으로 필터의 가중치와 편향을 학습합니다. 이번에는 역전파 구현, 모델 학습, 미래 기온 예측, 성능 튜닝을 요구하지 않습니다. ReLU는 1장 2강에서 배운 내용을 그대로 사용합니다.

## 실습 구성

| 구분 | 문제 | 할 일 |
|---|---|---|
| 실습 1 | 1-1 | 두 필터로 특징 맵을 만들고 첫 값을 손계산하기 |
| 실습 2 | 2-1 | ReLU 후 Max Pooling·Average Pooling 비교하기 |
| 실습 2 | 2-2 | CNN 이미지 분류 구조와 각 단계의 크기 이해하기 |
| 기초 과제 | 3-1 | 다음 8개 기온에 같은 합성곱·풀링 적용하기 |

**실습 2개 + 기초 과제 1개**이며, 마지막에는 **실습 마무리 복습 질문 6개**가 있습니다. 과제는 실습에서 사용한 함수와 순서를 그대로 적용하는 수준입니다.

## 시작 전에 읽을 용어

| 용어 | 쉽게 풀어쓴 뜻 | 이번 실습의 예 |
|---|---|---|
| CNN | 주변 값들의 패턴을 필터로 추출하는 신경망 | 기온의 작은 구간을 차례로 관찰 |
| Convolution, 합성곱 | 작은 구간과 필터를 대응 위치끼리 곱한 뒤 더하는 계산 | 기온 3개와 필터 값 3개를 곱하고 합침 |
| Filter, 필터·커널 | 어떤 패턴에 반응할지 정하는 가중치 묶음 | `[−1, 0, 1]` |
| Feature Map, 특징 맵 | 필터를 여러 위치에 적용해 얻은 결과 배열 | 한 필터로 만든 숫자 6개 |
| Stride, 이동 간격 | 다음 계산 위치로 얼마나 이동하는지 | 합성곱 1칸, 풀링 2칸 |
| Padding, 패딩 | 입력 가장자리에 값을 덧붙이는 것 | 이번에는 덧붙이지 않음 |
| ReLU | 음수를 0으로, 양수를 그대로 통과시키는 함수 | 변화량 중 양의 반응을 남김 |
| Pooling, 풀링 | 작은 구역을 대표값 하나로 줄이는 계산 | 2개씩 묶어 하나만 남김 |
| Max Pooling | 묶음의 최댓값을 남기는 풀링 | `[0.24, 0.26]`에서 0.26 |
| Average Pooling | 묶음의 평균을 남기는 풀링 | `[0.24, 0.26]`에서 0.25 |
| Channel, 채널 | 같은 위치에 대응하는 특징의 종류 | 필터 2개가 만드는 특징 맵 2개 |
| Flatten | 여러 차원의 값을 한 줄로 펼치는 단계 | 3×3×2개 값을 길이 18로 펼침 |
| Dense, 완전연결층 | 추출한 특징들을 연결해 출력값을 계산하는 층 | 앞 강의의 가중합으로 클래스별 점수 계산 |

### 계산 규칙

이번 합성곱은 **필터 길이 3, stride 1, padding 없음, bias 0**입니다. 필터를 뒤집지 않고 적힌 순서대로 계산합니다. 딥러닝에서 합성곱이라고 부르는 연산은 보통 이 방식(엄밀히는 교차상관)을 사용합니다. 용어 차이를 암기할 필요는 없습니다.

$$f_i=x_i k_0+x_{i+1}k_1+x_{i+2}k_2+0$$

`x`는 입력 기온, `k`는 필터, `i`는 계산을 시작하는 위치입니다. 길이 8에 길이 3 필터를 한 칸씩 적용하면 **8−3+1=6개** 결과가 생깁니다. 이 간단한 길이 계산은 stride 1이고 padding이 없을 때 사용합니다.

`[−1, 0, 1]`의 결과는 **세 번째 기온−첫 번째 기온**입니다. 두 관측 시점은 20분 떨어져 있습니다. 이는 20분 동안의 기온 차이이며, 시간을 나눈 변화율이나 미래 예측값은 아닙니다.


## 실습 준비 — 제공 코드를 그대로 실행하세요

첫 셀은 필요한 데이터만 읽습니다. `iloc[:8]`은 처음 8행, `iloc[8:16]`은 9번째부터 16번째 행입니다. Python의 위치 번호는 0부터 시작하며 끝 번호는 포함하지 않습니다.

두 번째 셀의 함수는 계산을 도와주는 **제공 함수**입니다. 함수를 새로 작성하는 것이 문제는 아닙니다.

- `conv1d_valid(values, kernel)`: 필터가 들어갈 수 있는 모든 위치를 한 칸씩 이동합니다. 대응 위치를 곱한 뒤 `np.sum()`으로 합칩니다. 편향은 0입니다.
- `pool_pairs(values)`: 값을 두 개씩 겹치지 않게 묶어 최댓값 배열과 평균 배열을 순서대로 돌려줍니다. 이 수업에서는 길이가 짝수인 입력만 사용합니다.
- `max_values, avg_values = pool_pairs(...)`처럼 반환값 두 개를 각각 저장합니다.


### 코드 읽기 안내 — 처음 사용하는 기호부터 확인하기

| 표현 | 읽는 방법과 하는 일 |
|---|---|
| `# 설명` | 주석입니다. 사람이 읽는 설명이며 Python이 계산하지 않습니다. |
| `x = ...` | 오른쪽 결과를 계산한 뒤 `x`라는 이름으로 보관합니다. 수학의 등식 증명과는 다릅니다. |
| `[...]` | 여러 값을 순서대로 적거나, 배열에서 필요한 부분을 선택할 때 씁니다. |
| `np.array([값1, 값2, 값3])` | 여러 숫자를 계산 가능한 NumPy 배열로 만듭니다. 원소는 배열 안의 숫자 하나를 뜻합니다. |
| `함수(입력)` | 미리 정해 둔 처리 과정에 입력을 넣어 실행합니다. `print(x)`는 x를 화면에 표시합니다. |
| `len(x)` | x에 들어 있는 원소 수를 셉니다. 값들의 합을 구하는 함수가 아닙니다. |
| `x[a:b]` | 위치 a부터 b 바로 전까지 선택합니다. Python의 첫 위치 번호는 0입니다. |
| `def 함수이름(...):` | 다음에 사용할 처리 방법을 정의합니다. 정의만으로 입력 데이터의 계산이 끝나는 것은 아닙니다. |
| `for ...:` | 정해진 위치를 차례로 바꾸며 같은 일을 반복합니다. 안으로 들여쓴 줄들이 반복됩니다. |
| `return 결과` | 함수가 계산한 결과를 호출한 곳으로 돌려줍니다. 화면에 출력하는 `print`와 역할이 다릅니다. |

**실행 순서:** 데이터 읽기 → 제공 함수 정의 → 실습 1 → 실습 2 → 과제입니다. 셀은 노트북에서 한 번에 실행하는 상자입니다. 앞 셀에서 만든 변수와 함수는 뒤 셀에서 사용합니다. 실행 순서를 건너뛰면 `x`나 `conv1d_valid`를 찾지 못할 수 있습니다.

**함수의 입력과 출력:** `conv1d_valid(x, k_mean)`에서 x는 함수 안의 `values`로, k_mean은 `kernel`로 전달됩니다. 함수가 돌려준 배열을 `f_mean = ...`으로 받습니다. `max_map, avg_map = pool_pairs(activated)`는 함수가 돌려준 두 배열을 왼쪽 이름에 순서대로 저장합니다. 두 이름이 있으므로 두 번 실행하는 것이 아닙니다.

`pool_pairs`는 이번 수업에서 사용하는 **짝수 길이의 1차원 배열**을 대상으로 합니다. 모든 크기의 입력이나 이미지에 그대로 적용하는 범용 CNN 함수는 아닙니다.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("jena_climate_2009_2016.csv", nrows=16)
times = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
# 위치 0~7의 기온 8개를 실습용 숫자 배열로 만듭니다. 끝 번호 8은 포함하지 않습니다.
x = df["T (degC)"].iloc[:8].to_numpy(dtype=float)
# 위치 8~15의 기온 8개를 과제용 배열로 만듭니다. 끝 번호 16은 포함하지 않습니다.
x_home = df["T (degC)"].iloc[8:16].to_numpy(dtype=float)

# 행·열 개수와 선택한 날짜·기온을 화면에 표시합니다.
print("읽은 행과 열:", df.shape)
print(df[["Date Time", "T (degC)"]].to_string(index=False))
# 이웃한 시각 차이를 초 단위로 바꾼 뒤 60으로 나누어 분 단위로 확인합니다.
print("선택 구간 간격(분):", times.diff().dropna().dt.total_seconds().unique() / 60)
# 기온이 비어 있는 칸의 개수를 확인합니다.
print("선택 기온의 결측값 수:", df["T (degC)"].isna().sum())
print("실습 입력:", x)
print("과제 입력:", x_home)


FileNotFoundError: [Errno 2] No such file or directory: 'jena_climate_2009_2016.csv'

In [ ]:
# 함수 정의: 입력 배열 values와 필터 배열 kernel을 받아 특징 맵을 돌려줍니다.
def conv1d_valid(values, kernel):
    results = []  # 계산 결과를 차례로 모을 빈 목록입니다.
    # 길이 8의 입력과 길이 3의 필터라면 시작 위치 i는 0, 1, 2, 3, 4, 5입니다.
    for i in range(len(values) - len(kernel) + 1):
        window = values[i:i + len(kernel)]  # 현재 위치부터 필터 길이만큼 꺼냅니다.
        value = np.sum(window * kernel)  # 같은 위치끼리 곱하고, 그 결과를 모두 더합니다.
        results.append(value)  # 현재 구간의 결과 한 개를 목록 맨 뒤에 붙입니다.
    return np.array(results)  # 반복이 끝난 뒤 모든 결과를 NumPy 배열로 돌려줍니다.


# 값을 두 개씩 묶어 최댓값 배열과 평균 배열을 함께 돌려주는 함수입니다.
def pool_pairs(values):
    maximums = []  # 각 묶음의 최댓값을 모읍니다.
    averages = []  # 각 묶음의 평균을 모읍니다.
    for i in range(0, len(values), 2):  # 길이 6일 때 i는 0, 2, 4입니다.
        window = values[i:i + 2]  # 현재 위치부터 두 개를 꺼냅니다.
        maximums.append(np.max(window))  # 두 값 중 큰 값 하나를 저장합니다.
        averages.append(np.mean(window))  # 두 값을 더하고 2로 나눈 값을 저장합니다.
    return np.array(maximums), np.array(averages)  # 최댓값이 첫째, 평균이 둘째입니다.


print("합성곱 함수와 풀링 함수 준비 완료")


합성곱 함수와 풀링 함수 준비 완료


---

## 실습 1. Convolution과 특징 추출


### 문제 1-1. 같은 기온에 두 가지 필터 적용하기

#### 문제 설명

첫 8개 기온 `x`에 다음 필터를 각각 적용하세요. 두 필터 모두 같은 입력에서 독립적으로 출발합니다.

| 필터 | 값 | 살펴볼 패턴 |
|---|---|---|
| 평균 필터 `k_mean` | `[1/3, 1/3, 1/3]` | 주변 3개 기온의 평균 수준 |
| 차이 필터 `k_diff` | `[−1, 0, 1]` | 구간 끝 기온과 시작 기온의 차이 |

#### 요구사항

1. `np.array()`로 두 필터를 만드세요.
2. 제공 함수로 `f_mean`, `f_diff`를 구하고 두 특징 맵과 길이를 출력하세요.
3. 첫 구간 `x[:3]`에 각 필터를 곱한 값과 그 합을 손계산하세요.
4. 차이 특징 맵의 부호를 보고 각 구간에서 끝 기온이 시작보다 높은지 낮은지 설명하세요.

#### 작성 도움말

`conv1d_valid(x, k_mean)`처럼 입력과 필터를 넣습니다. `len()`은 값의 개수를 반환합니다. 두 번째 필터도 `x`에 적용해야 하며 첫 특징 맵에 이어서 적용하지 않습니다.

#### 해석 질문

**Q1.** 입력 기온, 필터, 특징 맵은 각각 무엇인가요? 길이 8인 입력의 특징 맵 길이가 6인 이유는 무엇인가요?

**Q2.** 두 필터의 첫 번째 출력값은 각각 얼마인가요? 곱셈과 덧셈 과정을 쓰세요.

**Q3.** 차이 특징 맵에서 음수와 양수는 무엇을 뜻하나요? 양수라면 세 관측값이 모두 연속해서 증가했다는 뜻인가요?

**Q4.** 같은 입력에 다른 필터를 적용하면 왜 결과가 달라지나요? 이번 필터와 실제 CNN에서 학습하는 필터의 차이도 설명하세요.

#### 제출 결과

두 특징 맵·길이, 첫 출력값 손계산, Q1~Q4 답변.


In [ ]:
# 두 필터를 np.array()로 만드세요.
k_mean = np.array([1/3, 1/3, 1/3])
k_diff = np.array([-1.0, 0.0, 1.0])

# 같은 x에 각 필터를 적용하세요.
f_mean = conv1d_valid(x, k_mean)
f_diff = conv1d_valid(x, k_diff)

# 두 특징 맵과 각각의 길이를 출력하세요.
print("입력 x        :", x, "| 길이:", len(x))
print("평균 필터 k_mean:", np.round(k_mean, 4))
print("차이 필터 k_diff:", k_diff)
print("\n평균 특징 맵 f_mean:", np.round(f_mean, 4), "| 길이:", len(f_mean))
print("차이 특징 맵 f_diff:", np.round(f_diff, 4), "| 길이:", len(f_diff))

# 첫 구간 손계산 확인
print("\n첫 구간 x[:3]:", x[:3])
print("평균 필터 곱:", np.round(x[:3] * k_mean, 4), "→ 합:", round(np.sum(x[:3] * k_mean), 4))
print("차이 필터 곱:", x[:3] * k_diff, "→ 합:", round(np.sum(x[:3] * k_diff), 4))

# 구간별 차이의 부호
print("\n[구간별 차이 해석]")
for i, v in enumerate(f_diff):
    trend = "끝이 시작보다 높음" if v > 0 else ("끝이 시작보다 낮음" if v < 0 else "같음")
    print(f"구간 {i}: x[{i}]={x[i]} → x[{i+2}]={x[i+2]} | 차이 {v:+.2f} → {trend}")


입력 x        : [-8.02 -8.41 -8.51 -8.31 -8.27 -8.05 -7.62 -7.62] | 길이: 8
평균 필터 k_mean: [0.3333 0.3333 0.3333]
차이 필터 k_diff: [-1.  0.  1.]

평균 특징 맵 f_mean: [-8.3133 -8.41   -8.3633 -8.21   -7.98   -7.7633] | 길이: 6
차이 특징 맵 f_diff: [-0.49  0.1   0.24  0.26  0.65  0.43] | 길이: 6

첫 구간 x[:3]: [-8.02 -8.41 -8.51]
평균 필터 곱: [-2.6733 -2.8033 -2.8367] → 합: -8.3133
차이 필터 곱: [ 8.02 -0.   -8.51] → 합: -0.49

[구간별 차이 해석]
구간 0: x[0]=-8.02 → x[2]=-8.51 | 차이 -0.49 → 끝이 시작보다 낮음
구간 1: x[1]=-8.41 → x[3]=-8.31 | 차이 +0.10 → 끝이 시작보다 높음
구간 2: x[2]=-8.51 → x[4]=-8.27 | 차이 +0.24 → 끝이 시작보다 높음
구간 3: x[3]=-8.31 → x[5]=-8.05 | 차이 +0.26 → 끝이 시작보다 높음
구간 4: x[4]=-8.27 → x[6]=-7.62 | 차이 +0.65 → 끝이 시작보다 높음
구간 5: x[5]=-8.05 → x[7]=-7.62 | 차이 +0.43 → 끝이 시작보다 높음


#### 답변 작성란

**수식 및 계산 과정**

- 작성:
  - 입력 x = [−8.02, −8.41, −8.51, −8.31, −8.27, −8.05, −7.62, −7.62] (℃)
  - 평균 필터 첫 출력: (−8.02)×⅓ + (−8.41)×⅓ + (−8.51)×⅓ = (−24.94)/3 ≈ **−8.3133**
  - 차이 필터 첫 출력: (−8.02)×(−1) + (−8.41)×0 + (−8.51)×1 = 8.02 + 0 − 8.51 = **−0.49**
  - f_mean ≈ [−8.3133, −8.41, −8.3633, −8.21, −7.98, −7.7633], 길이 6
  - f_diff = [−0.49, 0.10, 0.24, 0.26, 0.65, 0.43], 길이 6
  - 구간별 부호: 구간 0만 음수(끝 기온이 시작보다 낮음), 구간 1~5는 양수(끝 기온이 시작보다 높음)

**Q1. 입력 기온, 필터, 특징 맵은 각각 무엇인가요? 길이 8인 입력의 특징 맵 길이가 6인 이유는 무엇인가요?**

- 답변: 입력 기온은 10분 간격으로 관측한 기온 8개, 필터는 어떤 패턴에 반응할지 정하는 가중치 3개(예: [−1, 0, 1]), 특징 맵은 필터를 입력의 여러 위치에 한 칸씩 옮기며 곱하고 더한 결과 배열입니다. 길이 3인 필터가 들어갈 수 있는 시작 위치는 0, 1, 2, 3, 4, 5의 6곳뿐입니다(위치 6에서 시작하면 값이 2개만 남음). 그래서 stride 1, padding 없음일 때 8 − 3 + 1 = 6개 결과가 생깁니다.

**Q2. 두 필터의 첫 번째 출력값은 각각 얼마인가요? 곱셈과 덧셈 과정을 쓰세요.**

- 답변: 평균 필터는 (−8.02)×⅓ + (−8.41)×⅓ + (−8.51)×⅓ ≈ −2.6733 − 2.8033 − 2.8367 ≈ **−8.3133**으로, 세 기온의 평균입니다. 차이 필터는 (−8.02)×(−1) + (−8.41)×0 + (−8.51)×1 = 8.02 + 0 − 8.51 = **−0.49**로, 세 번째 기온 − 첫 번째 기온입니다.

**Q3. 차이 특징 맵에서 음수와 양수는 무엇을 뜻하나요? 양수라면 세 관측값이 모두 연속해서 증가했다는 뜻인가요?**

- 답변: 음수는 20분 뒤의 기온(구간 끝)이 구간 시작보다 낮았다는 뜻이고, 양수는 더 높았다는 뜻입니다. 첫 구간 −0.49는 20분 동안 0.49℃ 내려갔음을 뜻합니다. 양수라도 세 값이 모두 연속 증가했다는 뜻은 아닙니다. 가운데 값에는 0이 곱해져 결과에 영향을 주지 않기 때문입니다. 예를 들어 구간 1은 −8.41 → −8.51 → −8.31로 중간에 한 번 내려갔지만 끝과 시작만 비교하면 +0.10입니다.

**Q4. 같은 입력에 다른 필터를 적용하면 왜 결과가 달라지나요? 이번 필터와 실제 CNN에서 학습하는 필터의 차이도 설명하세요.**

- 답변: 합성곱은 입력 값에 필터 값을 곱해 더하는 계산이므로, 필터 값이 다르면 같은 구간에서도 강조되는 성질이 달라집니다. 평균 필터는 주변의 기온 수준(약 −8℃)을, 차이 필터는 끝과 시작의 변화량을 꺼냅니다. 이번 필터는 계산 원리를 확인하기 위해 사람이 정한 고정값입니다. 실제 CNN에서는 필터 가중치와 편향을 처음에 임의로 정한 뒤 손실을 줄이도록 역전파와 Optimizer로 학습하므로, 어떤 패턴을 찾을지를 데이터가 결정합니다.


---

## 실습 2. Pooling과 CNN 이미지 분류 구조


### 문제 2-1. 특징 맵을 ReLU와 Pooling으로 요약하기

#### 문제 설명

문제 1-1의 **차이 특징 맵 `f_diff`만** 사용합니다. 먼저 ReLU를 적용하고, 그 결과를 두 개씩 묶어 Max Pooling과 Average Pooling을 각각 수행하세요.

$$a_i=max(0,f_i)$$

두 값 `[u, v]`를 묶었을 때 Max Pooling은 `max(u, v)`, Average Pooling은 `(u+v)/2`입니다. **풀링 크기 2, stride 2, padding 없음**으로 고정합니다.

#### 요구사항

1. `np.maximum(0.0, f_diff)`로 ReLU 결과 `activated`를 만드세요.
2. `pool_pairs(activated)`로 `max_map`, `avg_map`을 받으세요. 두 풀링은 같은 activated에서 각각 출발합니다.
3. 합성곱 결과, ReLU 결과, 두 풀링 결과와 길이를 출력하세요.
4. 세 묶음을 직접 적고 각 묶음의 최댓값과 평균을 계산하세요.

#### 해석 질문

**Q1.** ReLU는 이번 차이 특징 맵에서 어떤 정보를 남기고 어떤 정보를 없애나요?

**Q2.** Max Pooling과 Average Pooling의 세 결과는 각각 얼마인가요? 길이는 왜 6에서 3이 되나요?

**Q3.** 합성곱 필터와 이 두 풀링의 역할은 어떻게 다른가요? 일반적인 Max/Average Pooling에 학습할 가중치가 있나요?

**Q4.** 풀링 결과만 보고 입력 특징 맵을 정확히 복원할 수 있나요? 값의 개수를 줄이면 항상 분류가 더 잘되나요?

#### 제출 결과

단계별 결과·길이, 세 묶음의 풀링 계산, Q1~Q4 답변.


In [ ]:
# f_diff에 ReLU를 적용하세요.
activated = np.maximum(0.0, f_diff)

# 제공 함수로 두 풀링 결과를 받으세요.
max_map, avg_map = pool_pairs(activated)

# 각 단계의 결과와 길이를 출력하세요.
print("합성곱 결과 f_diff :", np.round(f_diff, 4), "| 길이:", len(f_diff))
print("ReLU 결과 activated:", np.round(activated, 4), "| 길이:", len(activated))
print("Max Pooling       :", np.round(max_map, 4), "| 길이:", len(max_map))
print("Average Pooling   :", np.round(avg_map, 4), "| 길이:", len(avg_map))

print("\n[묶음별 계산]")
for j in range(0, len(activated), 2):
    pair = activated[j:j + 2]
    print(f"묶음 {j//2 + 1}: {np.round(pair, 4)} → 최댓값 {np.max(pair):.4f}, 평균 {np.mean(pair):.4f}")


합성곱 결과 f_diff : [-0.49  0.1   0.24  0.26  0.65  0.43] | 길이: 6
ReLU 결과 activated: [0.   0.1  0.24 0.26 0.65 0.43] | 길이: 6
Max Pooling       : [0.1  0.26 0.65] | 길이: 3
Average Pooling   : [0.05 0.25 0.54] | 길이: 3

[묶음별 계산]
묶음 1: [0.  0.1] → 최댓값 0.1000, 평균 0.0500
묶음 2: [0.24 0.26] → 최댓값 0.2600, 평균 0.2500
묶음 3: [0.65 0.43] → 최댓값 0.6500, 평균 0.5400


#### 답변 작성란

**수식 및 계산 과정**

- 작성:
  - ReLU: a = max(0, f) → [max(0, −0.49), max(0, 0.10), max(0, 0.24), max(0, 0.26), max(0, 0.65), max(0, 0.43)] = **[0, 0.10, 0.24, 0.26, 0.65, 0.43]**
  - 묶음 1 [0, 0.10]: 최댓값 **0.10**, 평균 (0 + 0.10)/2 = **0.05**
  - 묶음 2 [0.24, 0.26]: 최댓값 **0.26**, 평균 (0.24 + 0.26)/2 = **0.25**
  - 묶음 3 [0.65, 0.43]: 최댓값 **0.65**, 평균 (0.65 + 0.43)/2 = **0.54**
  - 길이 변화: 6(합성곱) → 6(ReLU) → 3(풀링)

**Q1. ReLU는 이번 차이 특징 맵에서 어떤 정보를 남기고 어떤 정보를 없애나요?**

- 답변: 양수인 차이(구간 끝 기온이 시작보다 높아진 상승 반응)는 크기 그대로 남기고, 음수인 차이(하강)는 모두 0으로 바꿉니다. 첫 구간의 −0.49는 0이 되어 ‘0.49℃ 하강했다’는 정보가 사라졌습니다. 0이 된 값에서는 원래 하강 폭이 얼마였는지, 또는 변화가 정확히 0이었는지 구분할 수 없습니다.

**Q2. Max Pooling과 Average Pooling의 세 결과는 각각 얼마인가요? 길이는 왜 6에서 3이 되나요?**

- 답변: Max Pooling은 [0.10, 0.26, 0.65], Average Pooling은 [0.05, 0.25, 0.54]입니다. 크기 2, stride 2로 6개 값을 겹치지 않게 2개씩 묶으면 묶음이 6 ÷ 2 = 3개 생기고, 묶음마다 대표값 하나만 남기므로 길이가 3이 됩니다.

**Q3. 합성곱 필터와 이 두 풀링의 역할은 어떻게 다른가요? 일반적인 Max/Average Pooling에 학습할 가중치가 있나요?**

- 답변: 합성곱 필터는 가중치를 곱해 더하면서 ‘변화량’ 같은 특정 패턴을 **찾아내는** 역할을 합니다. 풀링은 이미 만든 특징 맵의 작은 구역을 최댓값이나 평균 하나로 **요약**해 크기를 줄이는 역할을 합니다. Max/Average Pooling은 ‘가장 큰 값 고르기’, ‘평균 내기’라는 정해진 규칙만 적용하므로 일반적으로 학습할 가중치나 편향이 없습니다.

**Q4. 풀링 결과만 보고 입력 특징 맵을 정확히 복원할 수 있나요? 값의 개수를 줄이면 항상 분류가 더 잘되나요?**

- 답변: 복원할 수 없습니다. Max Pooling 결과 0.26만 보면 짝이 0.24였는지 0.10이었는지 알 수 없고, 평균 0.25도 [0.24, 0.26]과 [0.25, 0.25]처럼 여러 조합에서 나올 수 있습니다. 값을 줄이면 계산량이 줄고 위치의 작은 변화에 덜 민감해지는 장점이 있지만, 세부 정보도 함께 사라집니다. 너무 많이 줄이면 분류에 필요한 정보를 잃어 오히려 성능이 나빠질 수 있으므로 항상 좋아진다고 할 수 없습니다.


### 문제 2-2. CNN 기반 이미지 분류의 처리 흐름 연결하기

#### 문제 설명

이번 문제는 **코드 없이 설명하는 구조 문제**입니다. 기온 배열은 시간 순서의 1차원 입력이고 이미지는 높이와 너비가 있는 2차원 입력입니다. 필터를 이동하며 주변 특징을 찾는 원리는 두 경우에 공통입니다.

다음은 이미지 1장을 세 종류 중 하나로 분류하는 **설명용 CNN 구조**입니다. 실제 이미지 파일이나 학습 결과는 사용하지 않습니다.

- 입력: 높이 8, 너비 8, 채널 1인 흑백 이미지.
- 합성곱: 3×3 필터 2개, stride 1, padding 없음. 각 필터는 입력의 채널 전체를 읽습니다.
- 활성화: ReLU.
- 풀링: 각 특징 맵에 2×2 Max Pooling, stride 2, padding 없음.
- Flatten: 모든 특징 값을 순서대로 한 줄로 펼침.
- Dense: 클래스 A·B·C의 점수 3개를 출력.

이번 표는 이미지 **한 장**을 나타내므로 여러 장을 묶는 배치 차원은 생략합니다. 배열 표기 순서는 **높이×너비×채널**로 통일합니다.

#### 요구사항

1. 다음 항목을 올바른 순서로 배열하세요: `Dense → 입력 이미지 → Pooling → Convolution → Flatten → ReLU`.
2. 입력, 합성곱, ReLU, 풀링, Flatten, Dense의 출력 크기를 쓰세요.
3. 각 단계가 하는 일을 한 문장씩 설명하세요.
4. 설명용 출력 점수가 **[A: 0.2, B: 1.4, C: −0.3]**일 때 가장 높은 점수의 클래스를 고르세요. 이 숫자는 확률로 변환하지 않은 가상의 점수입니다.

#### 작성 도움말

합성곱 결과의 높이와 너비는 각각 `8−3+1`입니다. 필터 한 개가 특징 맵 하나를 만들므로 필터 2개면 출력 채널이 2개입니다. ReLU는 크기를 바꾸지 않습니다. 6×6 특징 맵을 2×2씩 겹치지 않게 묶으면 3×3이 됩니다. Flatten은 값의 개수를 유지합니다.

#### 해석 질문

**Q1.** 올바른 처리 순서와 각 단계의 출력 크기는 무엇인가요?

**Q2.** 합성곱·ReLU·풀링·Flatten·Dense는 각각 어떤 역할을 하나요?

**Q3.** 예시 점수에서 선택할 클래스는 무엇인가요? B의 점수 1.4를 140%의 확률로 해석해도 되나요?

**Q4.** Jena 기온 실습과 이미지 분류 구조는 무엇이 같고 무엇이 다른가요? 기온 특징 맵을 계산한 것만으로 이미지 분류 모델 학습까지 완료했다고 할 수 있나요?

#### 제출 결과

단계별 순서·크기·역할 표, 클래스 선택, Q1~Q4 답변.


#### 답변 작성란

**수식 및 계산 과정**

- 작성:

| 순서 | 단계 | 출력 크기 (높이×너비×채널) | 계산 근거 | 하는 일 |
|---:|---|---|---|---|
| 1 | 입력 이미지 | 8×8×1 | 주어진 흑백 이미지 | 픽셀 밝기 값을 모델에 전달합니다. |
| 2 | Convolution | 6×6×2 | 8−3+1 = 6, 필터 2개 → 채널 2 | 3×3 필터가 이동하며 선·모서리 같은 지역 패턴을 찾아 특징 맵 2개를 만듭니다. |
| 3 | ReLU | 6×6×2 | 원소별 적용이라 크기 유지 | 음수를 0으로 바꾸고 양의 반응만 남겨 비선형성을 더합니다. |
| 4 | Pooling | 3×3×2 | 6 ÷ 2 = 3, 채널 수 유지 | 2×2 구역마다 최댓값을 남겨 특징 맵을 요약하고 크기를 줄입니다. |
| 5 | Flatten | 18 | 3×3×2 = 18, 값 개수 유지 | 2차원 특징 맵들을 Dense가 받을 수 있도록 한 줄로 펼칩니다. |
| 6 | Dense | 3 | 클래스 A·B·C | 18개 특징에 가중치를 곱해 더하고 편향을 더해 클래스별 점수를 냅니다. |

  - 예시 점수 [A: 0.2, B: 1.4, C: −0.3] 중 최댓값은 1.4 → **클래스 B** 선택

**Q1. 올바른 처리 순서와 각 단계의 출력 크기는 무엇인가요?**

- 답변: 입력 이미지(8×8×1) → Convolution(6×6×2) → ReLU(6×6×2) → Pooling(3×3×2) → Flatten(18) → Dense(3)입니다.

**Q2. 합성곱·ReLU·풀링·Flatten·Dense는 각각 어떤 역할을 하나요?**

- 답변: 합성곱은 필터를 이동하며 주변 픽셀의 패턴(선, 모서리 등)을 찾아 특징 맵을 만듭니다. ReLU는 음수 반응을 0으로 만들고 양의 반응을 남겨 비선형 표현을 가능하게 합니다. 풀링은 작은 구역을 대표값으로 요약해 크기를 줄이고 작은 위치 변화에 덜 민감하게 합니다. Flatten은 여러 차원의 특징 맵을 계산 없이 한 줄로 펼칩니다. Dense는 펼친 특징 전체를 가중합해 클래스별 점수를 계산합니다.

**Q3. 예시 점수에서 선택할 클래스는 무엇인가요? B의 점수 1.4를 140%의 확률로 해석해도 되나요?**

- 답변: 가장 높은 점수 1.4를 가진 **클래스 B**를 선택합니다. 1.4는 확률로 변환하지 않은 점수일 뿐이므로 140%로 해석하면 안 됩니다. 확률은 0~1(0~100%) 사이여야 하며, C의 점수처럼 음수도 나올 수 있다는 점에서도 확률이 아님을 알 수 있습니다. 확률로 보려면 Softmax 같은 변환이 추가로 필요합니다.

**Q4. Jena 기온 실습과 이미지 분류 구조는 무엇이 같고 무엇이 다른가요? 기온 특징 맵을 계산한 것만으로 이미지 분류 모델 학습까지 완료했다고 할 수 있나요?**

- 답변: 같은 점은 필터를 이동하며 대응 위치끼리 곱해 더하는 합성곱, 그 뒤의 ReLU와 풀링으로 특징을 추출·요약하는 원리입니다. 다른 점은 기온은 시간 방향의 1차원 입력이라 길이 3 필터가 한 방향으로만 이동하고, 이미지는 높이·너비가 있는 2차원 입력이라 3×3 필터가 가로·세로로 이동하며 채널도 다룬다는 것입니다. 기온 특징 맵 계산은 고정 필터로 한 순전파 일부일 뿐이므로 이미지 분류 모델 학습을 완료했다고 할 수 없습니다. 이미지 데이터, 정답 클래스, 손실 계산, 역전파로 필터와 Dense 가중치를 학습하는 과정이 모두 빠져 있습니다.


---

## 기초 과제. 다른 기온 구간에 같은 계산 적용하기


### 문제 3-1. 합성곱부터 Max Pooling까지 이어서 계산하기

#### 문제 설명

준비 셀의 `x_home`은 9번째부터 16번째 관측값입니다. 실습과 동일하게 **차이 필터 `[−1, 0, 1]` → ReLU → Max Pooling** 순서로 계산하세요. 새 함수 작성이나 모델 학습은 필요하지 않습니다.

합성곱은 필터 길이 3, stride 1, padding 없음, bias 0입니다. 풀링은 크기 2, stride 2, padding 없음입니다.

#### 요구사항

1. `home_filter`를 만들고 `conv1d_valid(x_home, home_filter)`로 `home_feature`를 구하세요.
2. ReLU 결과 `home_relu`를 구하세요.
3. `pool_pairs(home_relu)`에서 첫 반환값을 `home_max`로 받으세요. 두 번째 반환값은 받아 두되 이번 과제에서 출력·해석하지 않아도 됩니다.
4. 입력, 합성곱, ReLU, Max Pooling의 값과 길이를 출력하세요.
5. 첫 합성곱 값과 세 풀링 결과를 손계산하세요.

#### 해석 질문

**Q1.** 첫 구간의 합성곱 결과는 얼마이며, 어떻게 계산하나요?

**Q2.** ReLU를 적용한 특징 맵과 Max Pooling 결과는 각각 무엇인가요? 0인 풀링 결과는 기온이 0℃라는 뜻인가요?

**Q3.** 네 단계의 길이는 어떻게 바뀌나요? 앞선 실습과 결과 숫자가 다른 이유는 무엇인가요?

**Q4.** 이미지 분류 구조에서는 이렇게 추출·요약한 특징을 어떤 단계로 연결해 클래스별 점수를 만드나요?

#### 제출 결과

단계별 배열·길이, 손계산, Q1~Q4 답변. 예측 성능 평가나 추가 모델 비교는 요구하지 않습니다.


In [ ]:
# 실습에서 사용한 코드의 입력을 x_home으로 바꿔 적용하세요.
home_filter = np.array([-1.0, 0.0, 1.0])
home_feature = conv1d_valid(x_home, home_filter)
home_relu = np.maximum(0.0, home_feature)
home_max, home_avg_unused = pool_pairs(home_relu)

# 네 단계의 값과 길이를 출력하세요.
print("입력 x_home   :", x_home, "| 길이:", len(x_home))
print("합성곱 결과   :", np.round(home_feature, 4), "| 길이:", len(home_feature))
print("ReLU 결과     :", np.round(home_relu, 4), "| 길이:", len(home_relu))
print("Max Pooling   :", np.round(home_max, 4), "| 길이:", len(home_max))

print("\n첫 구간:", x_home[:3], "×", home_filter, "→", round(np.sum(x_home[:3] * home_filter), 4))
for j in range(0, len(home_relu), 2):
    pair = home_relu[j:j + 2]
    print(f"묶음 {j//2 + 1}: {np.round(pair, 4)} → 최댓값 {np.max(pair):.4f}")


입력 x_home   : [-7.91 -8.43 -8.76 -8.88 -8.85 -8.83 -8.66 -8.66] | 길이: 8
합성곱 결과   : [-0.85 -0.45 -0.09  0.05  0.19  0.17] | 길이: 6
ReLU 결과     : [0.   0.   0.   0.05 0.19 0.17] | 길이: 6
Max Pooling   : [0.   0.05 0.19] | 길이: 3

첫 구간: [-7.91 -8.43 -8.76] × [-1.  0.  1.] → -0.85
묶음 1: [0. 0.] → 최댓값 0.0000
묶음 2: [0.   0.05] → 최댓값 0.0500
묶음 3: [0.19 0.17] → 최댓값 0.1900


#### 답변 작성란

**수식 및 계산 과정**

- 작성:
  - 입력 x_home = [−7.91, −8.43, −8.76, −8.88, −8.85, −8.83, −8.66, −8.66] (℃)
  - 첫 합성곱: (−7.91)×(−1) + (−8.43)×0 + (−8.76)×1 = 7.91 + 0 − 8.76 = **−0.85**
  - 전체 합성곱(세 번째 − 첫 번째): −8.76+7.91 = −0.85, −8.88+8.43 = −0.45, −8.85+8.76 = −0.09, −8.83+8.88 = 0.05, −8.66+8.85 = 0.19, −8.66+8.83 = 0.17
  - home_feature = [−0.85, −0.45, −0.09, 0.05, 0.19, 0.17]
  - ReLU: home_relu = [0, 0, 0, 0.05, 0.19, 0.17]
  - Max Pooling: [0, 0] → **0**, [0, 0.05] → **0.05**, [0.19, 0.17] → **0.19** → home_max = [0, 0.05, 0.19]

**Q1. 첫 구간의 합성곱 결과는 얼마이며, 어떻게 계산하나요?**

- 답변: −0.85입니다. 첫 구간 [−7.91, −8.43, −8.76]과 필터 [−1, 0, 1]을 같은 위치끼리 곱하면 7.91, 0, −8.76이고, 이를 더하면 −0.85입니다. 9번째 관측에서 11번째 관측까지 20분 동안 기온이 0.85℃ 내려갔다는 뜻입니다.

**Q2. ReLU를 적용한 특징 맵과 Max Pooling 결과는 각각 무엇인가요? 0인 풀링 결과는 기온이 0℃라는 뜻인가요?**

- 답변: ReLU 결과는 [0, 0, 0, 0.05, 0.19, 0.17], Max Pooling 결과는 [0, 0.05, 0.19]입니다. 풀링 결과 0은 기온이 0℃라는 뜻이 아닙니다. 실제 기온은 약 −8℃입니다. 0은 ‘그 묶음에서 구간 끝 기온이 시작보다 높아지는 상승 반응이 없었다’는 뜻으로, 원래 차이 −0.85와 −0.45가 ReLU에서 0이 된 결과입니다.

**Q3. 네 단계의 길이는 어떻게 바뀌나요? 앞선 실습과 결과 숫자가 다른 이유는 무엇인가요?**

- 답변: 입력 8 → 합성곱 6 (8−3+1) → ReLU 6 (크기 유지) → Max Pooling 3 (6÷2)입니다. 필터, ReLU, 풀링 규칙은 실습과 똑같고 입력 기온 구간만 바뀌었기 때문에 숫자가 달라졌습니다. 이번 구간은 초반에 기온이 계속 내려가 음수 차이가 많았고, 그래서 ReLU 후 0이 3개 생겼습니다.

**Q4. 이미지 분류 구조에서는 이렇게 추출·요약한 특징을 어떤 단계로 연결해 클래스별 점수를 만드나요?**

- 답변: 풀링까지 거친 특징 맵을 **Flatten**으로 한 줄로 펼친 뒤, **Dense(완전연결층)**에 넣어 각 특징에 가중치를 곱해 더하고 편향을 더해 클래스별 점수를 계산합니다. 문제 2-2 구조라면 3×3×2 특징 → Flatten 18개 → Dense 3개 점수입니다. 가장 높은 점수의 클래스를 선택하고, 확률이 필요하면 Softmax 등으로 변환합니다.


## 제출 전 확인

- 실습 1과 실습 2, 기초 과제의 요구사항을 모두 확인했나요?
- 코드 출력과 손계산을 비교했나요?
- 문제마다 Q1~Q4에 답했나요?
- 기온·특징 맵·클래스 점수의 의미를 구분해서 설명했나요?


---

## 실습 마무리

실습 1·실습 2·기초 과제에서 다룬 내용을 자신의 말로 정리하세요. 새로운 심화 문제가 아니라 **배운 개념과 계산 과정을 확인하는 복습 질문**입니다.

1. Convolution, Filter, Feature Map은 각각 무엇인가요? 이번 기온 실습의 예를 하나씩 들어 설명하세요.

2. 입력 길이 8, 필터 길이 3, stride 1, padding 없음일 때 출력 길이는 왜 6인가요? 두 필터를 적용하면 무엇이 두 개 생기나요?

3. 차이 필터 [−1, 0, 1]의 결과가 양수라는 것은 무엇을 뜻하나요? ReLU에서 0이 되었다면 원래 차이를 정확히 알 수 있나요?

4. [0.24, 0.26]에 Max Pooling과 Average Pooling을 적용한 결과를 각각 계산하세요. 풀링이 남기는 정보와 잃는 정보도 설명하세요.

5. 이미지 분류 예시의 처리 순서와 출력 크기를 적으세요. 입력은 8×8×1, 합성곱은 3×3 필터 2개, 풀링은 2×2이고, 나머지 설정은 문제 2-2와 같습니다. Flatten과 Dense는 어떻게 다른가요?

6. 이번 실습에서 이미지 분류 모델을 학습했나요? 기온 실습과 이미지 CNN의 공통점·차이점, 고정 필터와 학습되는 필터의 차이를 설명하세요.


### 실습 마무리 답변 작성란

**1. Convolution, Filter, Feature Map은 각각 무엇인가요? 이번 기온 실습의 예를 하나씩 들어 설명하세요.**

- 답변: Convolution은 입력의 작은 구간과 필터를 대응 위치끼리 곱한 뒤 더하는 계산이고, Filter는 어떤 패턴에 반응할지 정하는 가중치 묶음이며, Feature Map은 필터를 여러 위치에 적용한 결과 배열입니다.

- 계산 과정 또는 예시: 필터 [−1, 0, 1]을 첫 구간 [−8.02, −8.41, −8.51]에 적용하는 계산이 Convolution(8.02 + 0 − 8.51 = −0.49)이고, [−1, 0, 1]이 Filter이며, 6개 위치의 결과 [−0.49, 0.10, 0.24, 0.26, 0.65, 0.43]이 Feature Map입니다.

**2. 입력 길이 8, 필터 길이 3, stride 1, padding 없음일 때 출력 길이는 왜 6인가요? 두 필터를 적용하면 무엇이 두 개 생기나요?**

- 답변: 길이 3 필터가 입력 안에 완전히 들어갈 수 있는 시작 위치가 0~5의 6곳뿐이기 때문입니다. 필터 하나가 특징 맵 하나를 만들므로, 두 필터를 적용하면 길이 6인 **특징 맵이 두 개**(채널 2개) 생깁니다.

- 계산 과정 또는 예시: 8 − 3 + 1 = 6. 실습 1-1에서 f_mean과 f_diff, 두 특징 맵이 각각 길이 6으로 만들어졌습니다.

**3. 차이 필터 [−1, 0, 1]의 결과가 양수라는 것은 무엇을 뜻하나요? ReLU에서 0이 되었다면 원래 차이를 정확히 알 수 있나요?**

- 답변: 양수는 구간의 세 번째 기온이 첫 번째 기온보다 높다는 뜻, 즉 20분 동안 기온이 올랐다는 뜻입니다(가운데 값은 반영되지 않음). ReLU에서 0이 되었다면 원래 값이 0 이하였다는 것만 알 수 있고, 정확히 얼마였는지는 알 수 없습니다.

- 계산 과정 또는 예시: 실습의 −0.49와 과제의 −0.85, −0.45, −0.09는 모두 ReLU 후 0이 되어, 0만 보고는 서로 구분할 수 없습니다.

**4. [0.24, 0.26]에 Max Pooling과 Average Pooling을 적용한 결과를 각각 계산하세요. 풀링이 남기는 정보와 잃는 정보도 설명하세요.**

- 답변: Max Pooling은 0.26, Average Pooling은 0.25입니다. Max Pooling은 구역에서 가장 강한 반응을, Average Pooling은 구역의 평균적인 반응 수준을 남깁니다. 반면 구역 안의 개별 값, 강한 반응이 정확히 어느 위치였는지, 값들 사이의 차이 같은 세부 정보는 잃습니다.

- 계산 과정 또는 예시: max(0.24, 0.26) = 0.26, (0.24 + 0.26) / 2 = 0.50 / 2 = 0.25.

**5. 이미지 분류 예시의 처리 순서와 출력 크기를 적으세요. 입력은 8×8×1, 합성곱은 3×3 필터 2개, 풀링은 2×2이고, 나머지 설정은 문제 2-2와 같습니다. Flatten과 Dense는 어떻게 다른가요?**

- 답변: 입력 8×8×1 → Convolution 6×6×2 → ReLU 6×6×2 → Max Pooling 3×3×2 → Flatten 18 → Dense 3입니다. Flatten은 가중치 없이 값의 배열 모양만 한 줄로 바꾸고 값의 개수(18)를 유지합니다. Dense는 그 18개 값에 학습되는 가중치를 곱해 더하고 편향을 더해 새로운 값(클래스 점수 3개)을 계산합니다.

- 계산 과정 또는 예시: 8 − 3 + 1 = 6, 6 ÷ 2 = 3, 3 × 3 × 2 = 18.

**6. 이번 실습에서 이미지 분류 모델을 학습했나요? 기온 실습과 이미지 CNN의 공통점·차이점, 고정 필터와 학습되는 필터의 차이를 설명하세요.**

- 답변: 학습하지 않았습니다. 고정 필터로 기온 특징 맵을 계산하고 이미지 분류 구조를 설명했을 뿐입니다. 공통점은 필터 이동 → 곱하고 더하기 → ReLU → 풀링으로 특징을 추출·요약하는 원리이고, 차이점은 기온은 시간 방향 1차원, 이미지는 높이·너비 2차원(채널 포함)이라는 점입니다. 고정 필터는 사람이 정한 값([−1, 0, 1] 등)이라 찾는 패턴이 미리 정해져 있고, 학습되는 필터는 정답이 있는 데이터로 손실을 줄이도록 역전파와 Optimizer가 값을 조정해 분류에 필요한 패턴을 스스로 찾습니다.

- 계산 과정 또는 예시: 이번 실습에서 필터 [1/3, 1/3, 1/3]과 [−1, 0, 1]은 처음부터 끝까지 값이 바뀌지 않았습니다.
